#  Spark Performance Optimization



In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark import StorageLevel
import time

spark = (
    SparkSession.builder
    .appName("Performance-Optimization")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)
spark.version

In [28]:
spark

##  Partitioning 



In [29]:
df = spark.range(0, 1_000_000).withColumn("value", (F.rand(seed=1) * 1000))

df.rdd.getNumPartitions()

df.write.mode("overwrite").parquet("s3a://matrix/lesson23/partitioning_demo")


In [30]:
df.rdd.getNumPartitions()

8

In [32]:
df = spark.range(0, 1_000_000).withColumn("value", (F.rand(seed=1) * 1000))

df.rdd.getNumPartitions()
print(df.rdd.getNumPartitions())

df =df.repartition(2)

df.write.mode("overwrite").parquet("s3a://matrix/lesson23/partitioning_demo_re")



8


## 2. Repartition vs Coalesce 


In [31]:
raw = spark.range(0, 1000, numPartitions=1000)   

def expand(idx, _iterator):
    n = 10_000 if idx < 500 else 10   
    return iter([(idx, i) for i in range(n)])

skewed_rdd = raw.rdd.mapPartitionsWithIndex(expand)
skewed_df = spark.createDataFrame(skewed_rdd, ["partition_id", "row_num"])

print( skewed_df.rdd.getNumPartitions())
print(skewed_df.count())

counts = skewed_df.rdd.glom().map(len).collect()


1000
5005000


In [33]:
coalesced = skewed_df.coalesce(2)
coalesced.write.mode("overwrite").parquet("s3a://matrix/lesson23/coalesce_2")



In [36]:
repartitioned = skewed_df.repartition(2)
repartitioned.write.mode("overwrite").parquet("s3a://matrix/lesson23/repartition_2")



## 3. JDBC Partitioned 



In [37]:
jdbc_url = "jdbc:postgresql://postgres:5432/lesson23"
conn_props = {
    "user": "matrix",
    "password": "matrix123",
    "driver": "org.postgresql.Driver",
}

In [38]:
t0 = time.time()
df_plain = spark.read.jdbc(url=jdbc_url, table="orders", properties=conn_props)
print("Partition :", df_plain.rdd.getNumPartitions())
print(" count:", df_plain.count())
print(f"time: {time.time() - t0:.2f} san.")

Partition : 1
 count: 2000000
time: 0.60 san.


In [39]:
t0 = time.time()
df_parallel = spark.read.jdbc(
    url=jdbc_url,
    table="orders",
    column="order_id",
    lowerBound=1,
    upperBound=2_000_000,
    numPartitions=8,
    properties=conn_props,
)
print("Partition :", df_parallel.rdd.getNumPartitions())
print("count:", df_parallel.count())
print(f"time: {time.time() - t0:.2f} san.")

Partition : 8
count: 2000000
time: 0.38 san.


## 4. Cache & Persist


In [40]:
big_df = spark.range(0, 5_000_000).withColumn("value", F.rand(seed=2) * 1000)



In [41]:
big_df.cache()
big_df.count() 

t0 = time.time()
big_df.agg(F.sum("value")).collect()
print(f"1-ci aggregation (cache ilə): {time.time() - t0:.2f} san.")
t0 = time.time()
big_df.agg(F.avg("value")).collect()
print(f"2-ci aggregation (cache ilə): {time.time() - t0:.2f} san.")

big_df.unpersist()

1-ci aggregation (cache ilə): 0.33 san.
2-ci aggregation (cache ilə): 0.19 san.


DataFrame[id: bigint, value: double]

## 9. Predicate Pushdown 


In [26]:
spark.range(0, 1_000_000).withColumn("amount", F.rand() * 1000) \
    .write.mode("overwrite").parquet("s3a://matrix/lesson23/pushdown_demo")

pushdown_df = spark.read.parquet("s3a://matrix/lesson23/pushdown_demo").filter(F.col("amount") > 500)
pushdown_df.explain()


In [ ]:
spark.stop()

In [ ]:
df = spark.read.jdbc(
url=jdbc_url, table="select 8 from where name='dndsjnc'",
column="order_id",
lowerBound=1, upperBound=100,
numPartitions=4,
properties=conn_props
)

In [ ]:
select * from orders where order_id >1 and orders <25
select * from orders where order_id >26 and orders <50
select * from orders where order_id >1 and orders  <25
